In [25]:
import heapq
import pandas as pd
import networkx as nx
import math
from itertools import count
from math import radians, cos, sin, asin, sqrt

# Dictionnaire global requis par votre fonction A*
STATION_COORDS = {}

# ==========================================
# 1. Votre Exact Dijkstra Implementation
# ==========================================
def dijkstra_with_path(graph, source, target):
    distances = {node: float('inf') for node in graph}
    distances[source] = 0
    parents = {node: None for node in graph}
    
    tie_breaker = count() 
    pq = [(0, next(tie_breaker), source)]
    
    while pq:
        current_time, _, current_node = heapq.heappop(pq)
        
        if current_node == target:
            break
            
        if current_time > distances[current_node]:
            continue
            
        for neighbor, travel_time in graph.get(current_node, []):
            new_time = current_time + travel_time
            
            if new_time < distances.get(neighbor, float('inf')):
                distances[neighbor] = new_time
                parents[neighbor] = current_node
                heapq.heappush(pq, (new_time, next(tie_breaker), neighbor))
                
    path = []
    if distances.get(target, float('inf')) != float('inf'):
        current = target
        while current is not None:
            path.append(current)
            current = parents[current]
        path.reverse()
        
    return path, distances[target]


# ==========================================
# 2. Votre Exact A* Implementation
# ==========================================
def a_star_search(graph, start, goal):
    open_set = []
    heapq.heappush(open_set, (0, start))
    
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0
    
    came_from = {}
    
    while open_set:
        current_f, current = heapq.heappop(open_set)
        
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1], g_score[goal]
            
        for neighbor, weight in graph.get(current, []):
            tentative_g = g_score[current] + weight
            
            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                
                f_score = tentative_g + get_distance_estimate(neighbor, goal)
                heapq.heappush(open_set, (f_score, neighbor))
                
    return None, float('inf')


def get_distance_estimate(current, goal):
    station1 = current[0] if isinstance(current, tuple) else current
    station2 = goal[0] if isinstance(goal, tuple) else goal
    
    x1, y1 = STATION_COORDS[station1]
    x2, y2 = STATION_COORDS[station2]
    
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)


# ==========================================
# 3. Distance & Graph Building (Sécurisé)
# ==========================================
def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371 * c


def build_network_and_coords(filepath):
    global STATION_COORDS
    df = pd.read_csv(filepath, delimiter=';')
    
    # Étape 1 : Rendre chaque arrêt unique par sa commune pour casser les homonymes distants
    df['unique_node_name'] = df['stop_name'] + " (" + df['Nom_commune'] + ")"
    
    coord_agg = df.groupby('unique_node_name')[['stop_lon', 'stop_lat']].mean()
    for node_name, row in coord_agg.iterrows():
        STATION_COORDS[node_name] = (row['stop_lon'], row['stop_lat'])
        
    graph = {stop: [] for stop in STATION_COORDS.keys()}
    
    # Étape 2 : Connecter les stations le long d'une même ligne
    for (route_id, mode), group in df.groupby(['route_id', 'mode']):
        unique_route_stops = group.groupby('unique_node_name')[['stop_lon', 'stop_lat']].mean().reset_index()
        if len(unique_route_stops) < 2:
            continue
            
        if mode in ['Metro', 'Rail', 'RER']:
            speed_factor = 2.0  # ~30 km/h
        elif mode == 'Tramway':
            speed_factor = 3.5  # ~17 km/h
        else:
            speed_factor = 5.0  # ~12 km/h pour les bus
            
        G_local = nx.Graph()
        for i, row1 in unique_route_stops.iterrows():
            for j, row2 in unique_route_stops.iterrows():
                if i < j:
                    d = haversine_distance(row1['stop_lon'], row1['stop_lat'], 
                                           row2['stop_lon'], row2['stop_lat'])
                    G_local.add_edge(row1['unique_node_name'], row2['unique_node_name'], weight=d)
                    
        mst_local = nx.minimum_spanning_tree(G_local)
        
        for u, v, data in mst_local.edges(data=True):
            travel_time = round(data['weight'] * speed_factor, 2)
            graph[u].append((v, travel_time))
            graph[v].append((u, travel_time))
            
    # Étape 3 : CORRECTION CRITIQUE DES CORRESPONDANCES A PIED
    # On autorise les correspondances entre deux arrêts ayant le même nom UNIQUEMENT s'ils sont à moins de 300 mètres
    df_correspondances = df.groupby('stop_name')['unique_node_name'].unique()
    for stop_name, nodes in df_correspondances.items():
        if len(nodes) > 1:
            for i in range(len(nodes)):
                for j in range(i + 1, len(nodes)):
                    node_a = nodes[i]
                    node_b = nodes[j]
                    
                    lon1, lat1 = STATION_COORDS[node_a]
                    lon2, lat2 = STATION_COORDS[node_b]
                    
                    dist_km = haversine_distance(lon1, lat1, lon2, lat2)
                    
                    # 300 mètres max (0.3 km) pour valider une correspondance à pied entre bus/métro de la même zone
                    if dist_km <= 0.3:
                        walk_time = 4.0  # Pénalité de correspondance de 4 minutes
                        graph[node_a].append((node_b, walk_time))
                        graph[node_b].append((node_a, walk_time))
                        
    return graph


def clean_and_verify_input(input_name, available_nodes):
    cleaned = input_name.strip().lower()
    matches = [node for node in available_nodes if node.lower().startswith(cleaned)]
    if not matches:
        return None
    paris_matches = [m for m in matches if "(paris)" in m.lower()]
    return paris_matches[0] if paris_matches else matches[0]

# ==========================================
# 4. Interface Utilisateur
# ==========================================
if __name__ == "__main__":
    csv_file = 'arrets-lignes.csv'
    print("🔄 Génération du vrai réseau d'Île-de-France sans téléportations...")
    
    network_graph = build_network_and_coords(csv_file)
    all_nodes = list(network_graph.keys())
    print(f"✅ Réseau sécurisé ! {len(all_nodes)} stations uniques prêtes.\n")
    
    raw_start = input("Entrez la station de départ (ex: Tolbiac) : ")
    raw_end = input("Entrez la gares d'arrivée (ex: Argenteuil) : ")
    
    start_station = clean_and_verify_input(raw_start, all_nodes)
    end_station = clean_and_verify_input(raw_end, all_nodes)
    
    if not start_station or not end_station:
        print("\n❌ Une des gares est introuvable.")
    else:
        print(f"\n📍 Départ : {start_station}")
        print(f"📍 Arrivée : {end_station}\n" + "-"*50)
        
        dijkstra_path, dijkstra_time = dijkstra_with_path(network_graph, start_station, end_station)
        a_star_path, a_star_time = a_star_search(network_graph, start_station, end_station)
        
        if dijkstra_time == float('inf'):
            print("❌ Aucun itinéraire physique n'existe entre ces deux points dans la base actuelle.")
        else:
            print(f"🏆 [RÉSULTATS DIJKSTRA SÉCURISÉ] :")
            print(f"⏱️  Temps total : {dijkstra_time:.2f} minutes")
            print(f"🗺️  Chemin : {' -> '.join(dijkstra_path)}")

🔄 Génération du vrai réseau d'Île-de-France sans téléportations...
✅ Réseau sécurisé ! 19091 stations uniques prêtes.


❌ Une des gares est introuvable.
